In [ ]:
# ============================================================
#  环境配置
#  - Colab 用户：取消注释下方 Colab 区块
#  - 本地 Jupyter 用户：直接运行 Local 区块
# ============================================================

# ── Colab 环境（取消注释后运行） ──
# !pip install torch transformers scikit-learn matplotlib numpy -q

# ── 本地 Jupyter 环境 ──
import subprocess, sys

def _install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for pkg in ["torch", "transformers", "scikit-learn", "matplotlib", "numpy"]:
    _install(pkg)

In [ ]:
import math, random, hashlib, re
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from torch import nn
from transformers import AutoModelForCausalLM, AutoTokenizer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# GPT-3 代码实战：学习实现 vs 工程实现

基于论文 *Language Models are Few-Shot Learners*（Brown et al., 2020），
用 **情境学习（In-Context Learning）+ 数据工程** 任务演示 GPT-3 的核心思想。

本 Notebook 包含两条并行路径，围绕 **同一组 few-shot 分类/翻译示例** 展开：

| | 学习路径 (Learning) | 工程路径 (Engineering) |
|---|---|---|
| 目标 | 理解 GPT-3 为什么能 few-shot，以及训练数据为何关键 | 掌握工业级 `transformers` 推理方式 |
| 实现方式 | L2：关键模块演示（质量分类器、LSH、污染检测、因果注意力） | E1：预训练 GPT-2 / GPT 风格 API 做 inference-only |
| 代码量 | 中等，强调可解释性 | 很少，强调可运行性 |
| 适合场景 | 面试准备、原理论证、论文复现 | 工程验证、Prompt 实验、批量推理 |

> 两条路径不是两套无关的代码，而是同一套 GPT-3 思想在不同抽象层级上的表达：学习路径解释“为什么”，工程路径回答“怎么用”。

## Section i：论文背景与任务背景

**论文**：Brown et al., *Language Models are Few-Shot Learners*, 2020  
**模型定位**：GPT-3 是大规模 **decoder-only Transformer**，核心不是结构革命，而是把已有架构扩展到足够大的参数规模、数据规模与算力规模。  
**本 Notebook 聚焦两件事**：

1. **推理侧**：为什么不给模型更新参数，只在 Prompt 里放示例，也能完成任务。
2. **训练侧**：为什么 GPT-3 不只是“模型更大”，还需要质量过滤、近重复去重、污染检测。

为什么选这个任务：
- **few-shot 分类 / 翻译** 最能直接展示 GPT-3 的情境学习能力；
- **数据清洗流水线** 最能体现 GPT-3 在工程层面的真正门槛；
- 两者合起来，才接近 GPT-3 论文的完整贡献。

## Section ii：最小必要理论

只保留后续代码真正需要的公式：

### 1. 自回归语言建模

$$P(x_1,\dots,x_T)=\prod_{t=1}^{T}P(x_t\mid x_{<t})$$

训练时：最大化下一个 token 的对数概率。  
推理时：给定 Prompt，逐步生成下一个 token。

### 2. 缩放点积注意力

$$\mathrm{Attention}(Q,K,V)=\mathrm{softmax}\left(\frac{QK^T}{\sqrt{d_k}} + M\right)V$$

其中 $M$ 是因果 mask，保证当前位置不能看未来 token。

### 3. Jaccard 相似度与 MinHash 近似

$$J(A,B)=\frac{|A\cap B|}{|A\cup B|}$$

MinHash 用多个哈希函数近似 Jaccard，相同签名比例近似集合相似度。

### 4. 污染检测

把 benchmark 文本切成 n-gram 指纹库；若训练文档和 benchmark 的 n-gram 重叠率过高，就移除该训练文档。

> 范围边界：这里不会训练真正的 GPT-3，也不会复现 175B 参数；学习路径只演示关键机制，工程路径只做可运行的工业推理。

### 组件映射表

| 论文组件 | 学习路径覆盖 | 工程库/API 对应 | 状态 |
|---------|-------------|----------------|------|
| 自回归语言建模 | 用 Prompt 构造 + 因果注意力演示 next-token 约束 | `AutoModelForCausalLM.generate()` | Runnable |
| In-Context Learning | 手写 Zero/One/Few-Shot Prompt + 注意力权重可视化 | `AutoTokenizer` + `model.generate()` | Runnable |
| 质量分类器 | `TF-IDF + LogisticRegression` 复现过滤思路 | 工业训练数据管线内部完成 | Runnable |
| Fuzzy Deduplication | `Shingling + MinHash + LSH banding` | 工业离线数据处理系统 | Runnable |
| Contamination Removal | 手写 n-gram overlap 检测 | 工业 benchmark 去污染流程 | Runnable |
| Scaling Laws | 只做概念解释，不在本 Notebook 训练验证 | 训练基础设施层面体现 | Explain-only |
| KV-Cache / 高效生成 | 解释 generate 背后的缓存思想 | `use_cache=True`（库内部默认支持） | Explain-only |

## Section 1：数据准备

这里的数据不是用来“训练 GPT-3”，而是用来复现 GPT-3 论文强调的两类关键工程问题：

1. **什么样的文本值得进入训练集**；
2. **如何防止重复与 benchmark 泄漏污染评测结果**。

同时，few-shot 推理部分会复用一组小型翻译/情感任务样例，方便学习路径与工程路径横向对齐。

In [ ]:
# ── 数据：质量过滤示例语料 ──
high_quality = [
    "The theory of general relativity describes gravity as the curvature of spacetime.",
    "Photosynthesis converts light energy into chemical energy stored in glucose.",
    "The transformer architecture relies on self-attention instead of recurrence.",
    "Machine learning models generalize better when training data is diverse and clean.",
    "A decoder-only language model predicts the next token conditioned on previous tokens.",
    "Open-domain corpora require filtering because raw web data contains spam and duplication.",
] * 8

low_quality = [
    "BUY NOW buy now!!! free crypto giveaway click click click",
    "lol omg this is insane subscribe share smash like button now",
    "cheap pills limited offer visit link link link immediately",
    "you wont believe this secret trick revealed today today today",
    "free followers free traffic free money no effort no skill",
    "random spam text asdf qwer zxcv clickbait clickbait clickbait",
] * 8

texts = high_quality + low_quality
labels = [1] * len(high_quality) + [0] * len(low_quality)

fewshot_examples = {
    "translation": [
        ("sea otter", "loutre de mer"),
        ("cheese", "fromage"),
        ("hello", "bonjour"),
    ],
    "sentiment": [
        ("This movie is wonderful", "positive"),
        ("The service was terrible", "negative"),
        ("I love this product", "positive"),
        ("This was a disappointing meal", "negative"),
    ],
}

print(f"Quality dataset size: {len(texts)}")
print(f"Few-shot task keys: {list(fewshot_examples.keys())}")

In [ ]:
# ── 训练 / 测试划分，并验证数据形状 ──
X_train_text, X_test_text, y_train, y_test = train_test_split(
    texts,
    labels,
    test_size=0.25,
    random_state=SEED,
    stratify=labels,
)

vectorizer = TfidfVectorizer(max_features=300, stop_words="english")
X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

print(f"Train matrix shape: {X_train.shape}")
print(f"Test matrix shape:  {X_test.shape}")
print(f"Positive ratio in train: {np.mean(y_train):.2f}")
print(f"Feature count: {len(vectorizer.get_feature_names_out())}")

## Section 2：共享组件

两条路径共用一组超参数与工具函数：
- 学习路径用它们组织 few-shot Prompt、因果 mask 与可视化；
- 工程路径用同样的 Prompt 构造函数喂给 HuggingFace 模型。

这样做的目的，是让“学习路径”和“工程路径”对齐到同一个输入接口，而不是各写各的。

In [ ]:
# ── 超参数（集中管理） ──
D_MODEL = 16
NUM_HEADS = 2
MAX_LEN = 12
NUM_HASHES = 80
NUM_BANDS = 20
NGRAM_N = 3
GEN_MAX_NEW_TOKENS = 12

print({
    "D_MODEL": D_MODEL,
    "NUM_HEADS": NUM_HEADS,
    "MAX_LEN": MAX_LEN,
    "NUM_HASHES": NUM_HASHES,
    "NUM_BANDS": NUM_BANDS,
    "NGRAM_N": NGRAM_N,
    "GEN_MAX_NEW_TOKENS": GEN_MAX_NEW_TOKENS,
})

In [ ]:
def format_prompt(task_desc, examples, query, separator="=>"):
    parts = [task_desc.strip()] if task_desc else []
    for inp, out in examples:
        parts.append(f"{inp} {separator} {out}")
    parts.append(f"{query} {separator}")
    return "\n".join(parts)


def causal_mask(seq_len):
    return torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()


def build_ngram_index(texts, n=3):
    index = set()
    for text in texts:
        words = text.lower().split()
        for i in range(len(words) - n + 1):
            index.add(tuple(words[i:i+n]))
    return index


def contamination_score(train_text, ngram_index, n=3):
    words = train_text.lower().split()
    if len(words) < n:
        return 0.0
    grams = [tuple(words[i:i+n]) for i in range(len(words) - n + 1)]
    hits = sum(gram in ngram_index for gram in grams)
    return hits / len(grams)

print(format_prompt("Translate English to French:", fewshot_examples["translation"][:2], "apple"))
print("Causal mask shape:", causal_mask(6).shape)
print("Mask example:\n", causal_mask(6).int())

## Section iii：学习路径（L2：关键模块演示）

GPT-3 不适合在免费 Colab 上从零训练，因此学习路径选择 **L2：关键模块演示**。  
目标不是复现 175B 模型，而是把论文中的关键思想拆成几个可运行模块：

1. 质量分类器：决定什么数据值得保留；
2. LSH 去重：决定什么数据是近重复；
3. 污染检测：决定什么训练样本必须删除；
4. 因果注意力：解释 why few-shot 提示能在推理时起作用。

### 模块 1：质量分类器

角色：从原始网页中筛选更像“百科 / 书籍 / 高质量说明文”的文本。

直觉：GPT-3 论文不是把所有 Common Crawl 都拿来训练，而是先做质量过滤。  
这里用最小可运行版本复现这个思想：

- 输入：文本 TF-IDF 特征
- 输出：属于“高质量文本”的概率
- 作用：作为数据工程中的第一道门

在完整工业系统里，这一步会比这里复杂得多，但核心思想一致：**先过滤，再训练**。

In [ ]:
quality_clf = LogisticRegression(max_iter=200, random_state=SEED)
quality_clf.fit(X_train, y_train)

y_pred = quality_clf.predict(X_test)
y_prob = quality_clf.predict_proba(X_test)[:, 1]

print("=== Quality Filter Report ===")
print(classification_report(y_test, y_pred, target_names=["low", "high"]))
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))

probe_texts = [
    "The model uses causal self-attention to predict the next token.",
    "click here free money free money free money",
    "Benchmark contamination can invalidate evaluation results.",
]
probe_scores = quality_clf.predict_proba(vectorizer.transform(probe_texts))[:, 1]
for text, score in zip(probe_texts, probe_scores):
    status = "keep" if score >= 0.5 else "drop"
    print(f"[{score:.3f}] {status:>4} | {text}")

### 模块 2：LSH 去重

流程如下：

$$\text{text} \rightarrow \text{shingles} \rightarrow \text{MinHash signature} \rightarrow \text{banding} \rightarrow \text{candidate pairs}$$

为什么这样做：
- 暴力比较所有文档对是 $O(n^2)$；
- 先把文本转成 shingle 集合，再用 MinHash 压缩成短签名；
- 最后只比较被 banding 筛出来的候选相似对。

这正是 GPT-3 类大规模数据管线必须解决的问题：**重复数据会让模型记忆而不是泛化**。

In [ ]:
def shingling(text, k=3):
    words = text.lower().split()
    if len(words) < k:
        return {" ".join(words)} if words else set()
    return {" ".join(words[i:i+k]) for i in range(len(words) - k + 1)}


def jaccard(a, b):
    return len(a & b) / len(a | b) if (a or b) else 0.0


def minhash_signature(shingle_set, num_hashes=80):
    signature = []
    for i in range(num_hashes):
        min_hash = float("inf")
        for shingle in shingle_set:
            h = int(hashlib.md5(f"{shingle}_{i}".encode()).hexdigest(), 16)
            min_hash = min(min_hash, h)
        signature.append(min_hash)
    return np.array(signature)


def signature_similarity(sig1, sig2):
    return float(np.mean(sig1 == sig2))


def lsh_banding(signatures, num_bands=20):
    rows_per_band = len(next(iter(signatures.values()))) // num_bands
    candidate_pairs = set()
    for band_idx in range(num_bands):
        buckets = defaultdict(list)
        start = band_idx * rows_per_band
        end = start + rows_per_band
        for doc_id, sig in signatures.items():
            bucket_key = tuple(sig[start:end])
            buckets[bucket_key].append(doc_id)
        for docs in buckets.values():
            if len(docs) > 1:
                for i in range(len(docs)):
                    for j in range(i + 1, len(docs)):
                        candidate_pairs.add((docs[i], docs[j]))
    return candidate_pairs

sample_docs = {
    "doc1": "the cat sat on the mat and looked around",
    "doc2": "the cat sat on the mat and looked up",
    "doc3": "large language models require clean and deduplicated corpora",
    "doc4": "large language models require clean deduplicated corpora for training",
    "doc5": "quantum computing uses superposition and entanglement",
}

sample_shingles = {k: shingling(v, k=2) for k, v in sample_docs.items()}
sample_sigs = {k: minhash_signature(v, num_hashes=NUM_HASHES) for k, v in sample_shingles.items()}
candidates = sorted(lsh_banding(sample_sigs, num_bands=NUM_BANDS))

print("Candidate pairs:", candidates)
for a, b in candidates:
    jac = jaccard(sample_shingles[a], sample_shingles[b])
    approx = signature_similarity(sample_sigs[a], sample_sigs[b])
    print(f"{a} vs {b} | Jaccard={jac:.3f} | MinHash≈{approx:.3f}")

### 模块 3：污染检测

训练集和测试集如果高度重叠，评测就不可信。  
GPT-3 论文显式做了 contamination removal。

这里的最小复现方式：
- 把 benchmark 文本切成 $n$-gram 指纹；
- 扫描训练文本；
- 如果重叠率过高，就标记为污染样本。

这一步不是提升分数，而是**防止作弊分数**。

In [ ]:
benchmark_texts = [
    "What is the capital of France",
    "Explain the theory of relativity",
]
ngram_index = build_ngram_index(benchmark_texts, n=NGRAM_N)

train_candidates = [
    "What is the capital of France and why is Paris famous",
    "The Eiffel Tower is located in Paris",
    "Please explain the theory of relativity clearly",
    "Neural language models scale with data and compute",
]

print("=== Contamination Check ===")
for text in train_candidates:
    score = contamination_score(text, ngram_index, n=NGRAM_N)
    flag = "contaminated" if score >= 0.3 else "safe"
    print(f"[{score:.2%}] {flag:>12} | {text}")

### 模块 4：因果注意力与 few-shot 为什么有效

下面不训练语言模型，只演示 **decoder-only self-attention** 的核心约束：

- 输入 shape：`(batch, seq, d_model)`
- 注意力分数 shape：`(batch, heads, seq, seq)`
- 因果 mask：上三角为 1，表示“未来不可见”

这能解释为什么把示例写进 Prompt 后，模型在生成最后一个 query 的答案时，会重点关注前面示例的输入输出格式。

In [ ]:
torch.manual_seed(SEED)
embedding = nn.Embedding(32, D_MODEL)
mha = nn.MultiheadAttention(embed_dim=D_MODEL, num_heads=NUM_HEADS, batch_first=True)

# toy token ids: (batch=1, seq=6)
input_ids = torch.tensor([[1, 5, 7, 9, 4, 2]])
x = embedding(input_ids)  # (1, 6, d_model)
mask = causal_mask(input_ids.shape[1]).to(x.device)  # (6, 6)

attn_out, attn_weights = mha(x, x, x, attn_mask=mask, need_weights=True, average_attn_weights=False)

print("Input embedding shape:", x.shape)
print("Attention output shape:", attn_out.shape)
print("Attention weights shape:", attn_weights.shape)  # (batch, heads, seq, seq)
print("Head 0 attention matrix:\n", attn_weights[0, 0].detach().numpy())

plt.figure(figsize=(5, 4))
plt.imshow(attn_weights[0, 0].detach().numpy(), cmap="viridis")
plt.title("Causal Self-Attention Head")
plt.xlabel("Key position")
plt.ylabel("Query position")
plt.colorbar()
plt.show()

### 训练 vs 推理的区别

GPT-3 的关键点之一，是**训练时和推理时行为不同**。

| 阶段 | 学习路径行为 | 工程路径行为 |
|------|------------|------------|
| 训练 | 用海量语料做 next-token prediction；靠因果 mask 保证不能看未来 | 这里不训练，仅解释原理 |
| 推理 | 把 few-shot 示例拼进 Prompt，让 query 位置关注前文模式 | `model.generate()` 做同样的自回归生成 |
| 参数更新 | 训练会更新权重 | 推理不更新权重 |
| 关键区别 | 学的是通用续写能力 | 用 Prompt 激活已有能力 |

核心洞见：**few-shot 不是在线学习新参数，而是利用已有参数，通过上下文选择正确的行为模式。**

## Section iv：工程路径（E1：成熟库 + inference-only）

根据可行性决策：

- **学习路径深度**：L2。GPT-3 本体无法在免费 Colab 稳定训练，但关键组件都能独立运行。
- **工程路径形式**：E1。HuggingFace `transformers` 提供成熟的 GPT 类推理接口。
- **工程动作**：inference-only。对 GPT-2 / GPT 风格模型，最稳定、最可运行的演示是直接做 Prompt 推理。

工程路径的目标是回答：论文里的 few-shot 思想，到了工业代码里到底怎么落地。

### 工程参数权衡

| 因素 | 增大时 | 吞吐量 | 内存 | 速度 | 效果 |
|------|--------|--------|------|------|------|
| batch_size | ↑ | ↑ | ↑↑ | ↑（单样本摊薄） | ~ |
| sequence_length | ↑ | ↓ | ↑↑ | ↓↓ | 可能 ↑ |
| max_new_tokens | ↑ | ↓ | ↑ | ↓ | 可能 ↑ |
| temperature | ↑ | ~ | ~ | ~ | 更随机 |
| use_cache | True | ~ | ↑ | ↑↑ | ~ |
| device | CPU→GPU | ↑↑ | GPU-bound | ↑↑↑ | ~ |

## 面试与项目表达

### 高频面试题

**Q1：GPT-3 的 few-shot learning 为什么不需要更新参数？**

- few-shot 发生在推理阶段，本质是上下文学习，不是参数学习。
- 模型参数保持冻结，变化的是 Prompt。
- 自注意力会把 query 和前面的示例模式关联起来。

**Q2：GPT-3 和 GPT-2 的本质差异是什么？**

- 两者都属于 decoder-only Transformer。
- GPT-3 的关键突破主要来自更大的参数、数据与算力规模。
- 同时数据工程更系统：过滤、去重、污染检测都更重要。

**Q3：为什么 GPT-3 要做去重和污染检测？**

- 去重是为了减少重复记忆，提升泛化。
- 污染检测是为了避免 benchmark 泄漏造成虚高结果。
- 这两者都直接影响评测可信度。

**Q4：`generate()` 本质在做什么？**

- 编码 Prompt；
- 计算当前上下文下最后位置的 logits；
- 选取下一个 token；
- 拼回上下文并重复。

**Q5：为什么说 GPT-3 更像“规模创新”而不是“结构创新”？**

- 因为核心结构仍然是 Transformer decoder。
- 真正变化是规模达到阈值后，few-shot 能力显著增强。
- 这与 scaling laws 的观察一致。

**Q6：few-shot、fine-tuning、instruction tuning 有什么区别？**

- few-shot：不改参数，只改上下文。
- fine-tuning：更新参数来适配任务。
- instruction tuning：用指令数据继续训练，让模型更会遵循指令。

### 面试速答提纲

| # | 问题 | 一句话回答 |
|---|------|-----------|
| 1 | few-shot 为什么有效？ | 因为注意力会从上下文示例里提取任务模式。 |
| 2 | GPT-3 强在哪？ | 强在规模扩展和更系统的数据工程。 |
| 3 | 为什么要做污染检测？ | 为了保证 benchmark 评测公平可信。 |
| 4 | `use_cache` 的作用？ | 缓存历史 KV，减少重复计算。 |
| 5 | 为什么做 batch inference？ | 提高吞吐量，降低单样本成本。 |

### 项目表达

> 如果面试官问“你做过什么相关项目”，可以这样组织回答：

- **学习深度**：我把 GPT-3 论文拆成了 few-shot 推理、质量过滤、LSH 去重、污染检测几个关键模块，并做了最小可运行复现。
- **工程能力**：我用 HuggingFace `transformers` 搭了 GPT 风格推理流程，覆盖单样本和 batch Prompt 推理。
- **对比思考**：我能解释手写模块和工业 API 的对应关系，比如 `generate()` 本质上就是封装好的自回归解码循环。

### 延伸阅读与对比

| 对比维度 | GPT-2 | GPT-3 | InstructGPT |
|---------|-----------|-----------|-----------|
| 核心思想 | 零样本 Prompt | few-shot / in-context learning | 指令对齐 |
| 训练重点 | 语言建模 | 规模扩展 + 数据工程 | SFT + RLHF |
| 适用场景 | 生成实验 | 通用任务迁移 | 对话助手 |

### 进阶探索方向

- **Scaling laws**：理解参数、数据、算力与 loss 的幂律关系。
- **KV-cache**：理解长文本生成为什么能靠缓存提速。
- **Prompt engineering**：继续比较 zero-shot、one-shot、few-shot 的差异。
- **Instruction tuning**：理解 GPT-3 之后模型为何更强调对齐。

In [ ]:
plt.figure(figsize=(6, 4))
labels_plot = ["quality_filter", "dedup", "decontam", "fewshot_infer"]
values_plot = [1.0, len(candidates) / len(sample_docs), 0.5, 1.0]
plt.bar(labels_plot, values_plot, color=["steelblue", "orange", "green", "purple"])
plt.title("GPT-3 Tutorial Coverage Summary")
plt.ylabel("Relative coverage")
plt.ylim(0, 1.2)
plt.grid(axis="y", alpha=0.3)
plt.show()

## Section vii：结果与边界

### 结果小结

- 学习路径说明了 GPT-3 的突破不仅来自规模，也来自数据工程与推理机制的配合。
- 工程路径说明了真实使用中通常直接调用成熟库，而不是自己重写完整大模型。

### 两条路径的边界

- **学习路径得到什么**：机制透明、适合论文讲解与面试。
- **学习路径失去什么**：不能代表真实 175B 模型效果。
- **工程路径得到什么**：立即可运行、便于做 Prompt 实验。
- **工程路径失去什么**：内部细节被高度封装。

### 实战建议

- 讲原理、准备面试：优先学习路径；
- 做实验、做产品验证：优先工程路径；
- 最理想状态：既能解释关键机制，也会用工业 API。

## Section vi：训练 vs 推理差异（双路径统一视角）

| 阶段 | 学习路径行为 | 工程路径行为 |
|------|------------|------------|
| 训练 | 海量语料上的 next-token prediction | 本 Notebook 不执行大模型训练 |
| 推理输入 | 手动组织 Prompt 与 few-shot 示例 | tokenizer 编码 Prompt |
| 推理循环 | 概念上逐 token 自回归 | `generate()` 自动循环 |
| 上下文利用 | 注意力读取前文示例模式 | 库内部 GPT block 完成同样事情 |
| 缓存 | 只解释原理 | `use_cache=True` 提升速度 |

> 同一个基础算法，在学习路径里被展开解释，在工程路径里被封装调用。

In [ ]:
comparison_rows = [
    ("zero-shot", format_prompt("Translate English to French:", [], "apple")),
    ("few-shot", format_prompt("Translate English to French:", fewshot_examples["translation"], "apple")),
]

for mode, prompt in comparison_rows:
    print("=" * 70)
    print(mode)
    print(prompt)
    print("Output:", hf_generate(prompt, max_new_tokens=8, temperature=0.2, do_sample=True))

## Section v：学习路径 vs 工程路径对比

| 对比维度 | 学习路径 | 工程路径 |
|---------|---------|---------|
| 教育价值 | 能拆开看清 few-shot、过滤、去重、污染检测各自作用 | 能快速掌握工业 API 的真实使用方式 |
| 代码量 | 中等，包含多个独立模块 | 少，核心在 `from_pretrained()` + `generate()` |
| 灵活性 | 高，可替换任意模块 | 中，受限于预训练模型与库接口 |
| 稳定性 | 演示代码清晰，但不具备工业鲁棒性 | 高，库实现经过大规模验证 |
| 工业适配度 | 更适合教学、论文讲解、原型理解 | 更适合真实推理服务、批处理、快速验证 |
| 适用场景 | 面试准备；理解 why GPT-3 有效 | Prompt 实验；快速推理；接入现有系统 |

In [ ]:
batch_prompts = [
    format_prompt("Translate English to French:", fewshot_examples["translation"][:2], "apple"),
    format_prompt("Translate English to French:", fewshot_examples["translation"][:2], "book"),
]

encoded_batch = tokenizer(batch_prompts, return_tensors="pt", padding=True, truncation=True).to(device)
input_lengths = encoded_batch["attention_mask"].sum(dim=1)

with torch.no_grad():
    batch_output = model.generate(
        **encoded_batch,
        max_new_tokens=8,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        use_cache=True,
    )

for i, generated in enumerate(batch_output):
    prompt_len = int(input_lengths[i].item())
    new_tokens = generated[prompt_len:]
    print(f"Prompt {i} output:", tokenizer.decode(new_tokens, skip_special_tokens=True).strip())

### Batch inference：padding、attention mask、吞吐量

工业推理常见的是 batch 而不是单条请求。  
关键点：
- `padding=True`：把不同长度 Prompt 对齐；
- `attention_mask`：区分真实 token 与 padding；
- batch 更高吞吐，但会增加显存 / 内存占用。

In [ ]:
translation_prompt = format_prompt(
    "Translate English to French:",
    fewshot_examples["translation"],
    "the cat"
)

sentiment_prompt = format_prompt(
    "Classify the sentiment as positive or negative:",
    fewshot_examples["sentiment"][:2],
    "The hotel room was clean and comfortable"
)

for title, prompt in [("translation", translation_prompt), ("sentiment", sentiment_prompt)]:
    print("=" * 70)
    print(title)
    print(prompt)
    print("Output:", hf_generate(prompt, max_new_tokens=10, temperature=0.2, do_sample=True))

### 组件对应关系

| 学习路径实现 | 工程库内部对应 | 说明 |
|---|---|---|
| `format_prompt()` | tokenizer 之前的输入组织 | Prompt 仍由使用者设计 |
| `causal_mask()` + 注意力演示 | GPT-2 block 内部 causal attention | 相同机制，更高抽象 |
| few-shot 示例拼接 | `model.generate()` 读取完整上下文 | 相同推理思想 |
| 污染检测 / 去重示例 | 预训练前数据管线 | 工程推理阶段不可见 |
| KV-cache 概念 | `use_cache=True` | 加速逐 token 生成 |

### Black-box 拆解：`generate()` 背后做了什么

`model.generate()` 内部仍然是标准的 decoder-only 自回归循环：

1. tokenizer 把 Prompt 编成 token ids；
2. 模型对完整上下文做 causal self-attention；
3. 取最后一个位置的 logits；
4. 选择下一个 token；
5. 继续把新 token 接回上下文；
6. 直到遇到 EOS 或达到长度上限。

因此，工程路径并没有换掉论文中的算法，只是把大量细节封装到了库里。

In [ ]:
def hf_generate(prompt, max_new_tokens=GEN_MAX_NEW_TOKENS, temperature=0.7, do_sample=True):
    encoded = tokenizer(prompt, return_tensors="pt", padding=False).to(device)
    generate_kwargs = dict(
        **encoded,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        use_cache=True,
    )
    if do_sample:
        generate_kwargs["temperature"] = temperature
    with torch.no_grad():
        output = model.generate(**generate_kwargs)
    prompt_len = encoded["input_ids"].shape[1]
    new_tokens = output[0, prompt_len:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

single_prompt = format_prompt("Translate English to French:", fewshot_examples["translation"][:2], "apple")
print(single_prompt)
print("\nModel output:", hf_generate(single_prompt, max_new_tokens=8, temperature=0.3, do_sample=True))

In [ ]:
MODEL_NAME = "openai-community/gpt2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
model.eval()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loaded model:", MODEL_NAME)
print("Pad token id:", tokenizer.pad_token_id)
print("EOS token id:", tokenizer.eos_token_id)
print("Parameter count (M):", round(sum(p.numel() for p in model.parameters()) / 1e6, 2))